# Day 13: Capstone Project — Build Your Own Production Agent 🏆

**Agentic AI Hands-On Course** | Dr. Kanthi Kiran Sirra | Sr. AI Engineer

**This notebook is a guided template. You fill in the TODO sections.**

Your agent must demonstrate all 6 mandatory capabilities:
1. ✅ LangGraph StateGraph (3+ nodes)
2. ✅ ChromaDB RAG (10+ documents)
3. ✅ Conversation memory (MemorySaver + thread_id)
4. ✅ Self-reflection (eval node or review loop)
5. ✅ Tool use (at least one tool beyond retrieval)
6. ✅ Deployment (Streamlit UI or FastAPI)

---
### Before you write any code — answer these three questions:
1. **What domain am I building for?** (e.g., HR Policy Bot, Study Buddy for Physics, Research Assistant)
2. **Who is the user?** (e.g., students asking questions, employees checking policies)
3. **What does success look like?** (e.g., agent answers 90% of domain questions faithfully)

Write your answers in the cell below before proceeding.

## My Capstone Plan

**Domain:** Cybersecurity — Threat Detection & Analysis

**User:** Security analysts, developers, and IT administrators who need fast, accurate answers about cyber threats, attack patterns, and defense strategies.

**Success looks like:** Agent answers ≥ 90% of cybersecurity questions faithfully (faithfulness ≥ 0.7), correctly admits when a question is out-of-scope, and detects active threat patterns (brute force, DDoS, phishing) using the tool node.

**Tool I will add:** threat_detector_tool — analyzes a question for threat signatures (Brute Force, DDoS, Phishing) and returns threat type + severity + recommended action

**Deployment choice:** Streamlit UI

---
## 0. Setup

In [1]:
# ============================================================
# COLAB USERS ONLY — Uncomment if using Google Colab
# ============================================================
# !pip install langgraph langchain-groq langchain-core chromadb \
#              sentence-transformers ragas ddgs python-dotenv -q

# from google.colab import userdata
# import os
# os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

In [2]:
import os
from dotenv import load_dotenv
load_dotenv()

from langchain_groq import ChatGroq
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import MemorySaver
from typing import TypedDict, List
import chromadb
from sentence_transformers import SentenceTransformer
from importlib.metadata import version

groq_key = os.getenv("GROQ_API_KEY", "").strip()
print(f"Groq API Key: {'✅ Loaded' if len(groq_key) > 10 else '❌ Missing'}")
print(f"LangGraph:    {version('langgraph')}")

try:
    llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)
    r = llm.invoke("Say ready in 1 word.")
    print(f"LLM:          ✅ {r.content}")
except Exception as e:
    print(f"LLM:          ⚠️ Warning - {type(e).__name__}: Continuing anyway...")
    llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)

Groq API Key: ✅ Loaded
LangGraph:    1.1.6
LLM:          ✅ Ready.


---
## Part 1 — Domain Setup: Knowledge Base

Load at least 10 documents about your domain. Write them as strings or load from files.

**Tips:**
- Each document should be 100-500 words
- Cover different aspects of your domain (don't repeat the same topic)
- Documents should be specific enough to answer concrete questions

In [3]:
from agent import DOCUMENTS, embedding_model, collection, llm

# ── Build ChromaDB ─────────────────────────────────────────
print("Loading embedding model...")

print(f"✅ Knowledge base ready: {collection.count()} documents")
for d in DOCUMENTS:
    print(f"   • {d['topic']}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading embedding model...
✅ Knowledge base ready: 11 documents
   • SQL Injection
   • Cross-Site Scripting (XSS)
   • Cross-Site Request Forgery (CSRF)
   • Distributed Denial of Service (DDoS)
   • Phishing Attacks
   • Malware and Trojans
   • Ransomware Attacks
   • Zero-Day Vulnerabilities
   • Firewalls and Intrusion Detection Systems
   • Incident Response Lifecycle
   • Brute Force and Credential Attacks


In [5]:
# ── Test retrieval before building the graph ──────────────
test_query = "How do I prevent SQL injection in my web application?"

q_emb   = embedding_model.encode([test_query])[0].tolist()
results = collection.query(query_embeddings=[q_emb], n_results=3, include=["documents", "metadatas"])

print(f"Query: {test_query}")
print(f"\nTop 3 retrieved chunks:")
for i, (doc, meta) in enumerate(zip(results["documents"][0], results["metadatas"][0])):
    print(f"\n[{i+1}] Topic: {meta['topic']}")
    print(f"    Text: {doc[:200]}...")

print("\n✅ If the retrieved chunks are relevant — retrieval is working correctly.")

Query: How do I prevent SQL injection in my web application?

Top 3 retrieved chunks:

[1] Topic: SQL Injection
    Text: SQL Injection is a code injection technique used by attackers to insert malicious SQL statements
into input fields of a web application. When the application executes the query, the injected SQL code ...

[2] Topic: Cross-Site Scripting (XSS)
    Text: Cross-Site Scripting (XSS) is a security vulnerability that allows attackers to inject malicious
scripts into web pages viewed by other users. These scripts execute in the victim's browser within the ...

[3] Topic: Brute Force and Credential Attacks
    Text: Brute force attacks systematically try many password and credential combinations to gain
unauthorized access to systems or accounts. Attack types: (1) Dictionary attacks using common passwords and
wor...

✅ If the retrieved chunks are relevant — retrieval is working correctly.


---
## Part 2 — State Design

**Design your State TypedDict BEFORE writing any node.** Every field a node needs must be a State field.

The template below is the base. Add domain-specific fields as needed.

In [6]:
from agent import SecurityXState

print("State imported successfully with fields:", list(SecurityXState.__annotations__.keys()))

State imported successfully with fields: ['question', 'messages', 'route', 'retrieved', 'sources', 'tool_result', 'answer', 'faithfulness', 'eval_retries', 'threat_type', 'severity']


---
## Part 3 — Node Functions

Write each node as a Python function. **Test each node in isolation before adding it to the graph.**

The mandatory nodes are scaffolded below. Add domain-specific nodes as needed.

In [7]:
from agent import memory_node, router_node, retrieval_node, skip_retrieval_node

# Quick test
test_state = {"question": "What is RAG?", "messages": []}
result = memory_node(test_state)
print(f"memory_node test: messages={result['messages']}")
print("✅ memory_node works")

memory_node test: messages=['User: What is RAG?']
✅ memory_node works


In [8]:
# Quick test
test_state2 = {"question": "What did you just say?", "messages": ["User: hi"], "route": "retrieve"}
result2 = router_node(test_state2)
print(f"router_node test: route='{result2['route']}'")
print("✅ router_node works")

router_node test: route='skip'
✅ router_node works


In [9]:
from agent import retrieval_node, skip_retrieval_node

# Quick test
test_state3 = {"question": "How do I prevent SQL injection?"}
result3 = retrieval_node(test_state3)
print(f"retrieval_node test: sources={result3['sources']}")
print(f"  Context preview: {result3['retrieved'][:200]}...")
print("✅ retrieval_node works")

retrieval_node test: sources=['SQL Injection', 'Cross-Site Scripting (XSS)', 'Brute Force and Credential Attacks']
  Context preview: [SQL Injection]
SQL Injection is a code injection technique used by attackers to insert malicious SQL statements
into input fields of a web application. When the application executes the query, the in...
✅ retrieval_node works


In [10]:
from agent import tool_node

# Test threat detection
test_threat = {"question": "We are seeing 500 failed login attempts per minute from multiple IPs"}
result = tool_node(test_threat)
print(f"tool_node test for brute force:")
print(result["tool_result"])
print(f"\nThreat type: {result['threat_type']}")
print(f"Severity: {result['severity']}")
print("✅ tool_node works")

tool_node test for brute force:
THREAT DETECTED: Brute Force Attack
Severity: Medium
Recommended Action: Monitor login attempts. Consider rate limiting on authentication endpoints.
Detection Basis: Login failure pattern analysis

Threat type: Brute Force
Severity: Medium
✅ tool_node works


In [11]:
from agent import answer_node

print("answer_node imported successfully")

answer_node imported successfully


In [12]:
from agent import eval_node, save_node

print("eval_node and save_node imported successfully")

eval_node and save_node imported successfully


---
## Part 4 — Graph Assembly

Connect your nodes. The routing functions decide which path to take.

In [13]:
from agent import build_agent_graph

app = build_agent_graph()

print("✅ Graph compiled successfully!")
print("Nodes: ['memory', 'router', 'retrieve/skip/tool', 'answer', 'eval', 'save']")

✅ Graph compiled successfully!
Nodes: ['memory', 'router', 'retrieve/skip/tool', 'answer', 'eval', 'save']


---
## Part 5 — Testing

Test with at least 10 questions including 2 red-team tests. Document each as PASS or FAIL.

In [14]:
from agent import ask

TEST_QUESTIONS = [
    {"q": "What is SQL injection and how can I prevent it in my web application?",
     "expect": "Should explain SQL injection and prevention from KB", "red_team": False},
    {"q": "How does Cross-Site Scripting (XSS) work and what are its three types?",
     "expect": "Should describe XSS types from KB", "red_team": False},
    {"q": "What are the main phases of incident response lifecycle?",
     "expect": "Should list 6 phases from KB", "red_team": False},
    {"q": "How do ransomware attacks work and what should I do if infected?",
     "expect": "Should describe ransomware lifecycle and response from KB", "red_team": False},
    {"q": "What is the difference between IDS and IPS?",
     "expect": "Should explain IDS vs IPS from Firewalls doc", "red_team": False},
    {"q": "We are seeing 500 failed login attempts per minute from multiple IPs",
     "expect": "Tool should detect Brute Force, High severity", "red_team": False},
    {"q": "Our server is experiencing a massive traffic spike and is becoming unreachable",
     "expect": "Tool should detect DDoS, Critical severity", "red_team": False},
    {"q": "What is a zero-day vulnerability and why is it dangerous?",
     "expect": "Should explain zero-day from KB", "red_team": False},
    {"q": "What mitigation strategies did you mention for the brute force scenario?",
     "expect": "Should reference earlier answer about brute force in same thread", "red_team": False},
    {"q": "What is the best recipe for making pasta carbonara?",
     "expect": "Should admit it does not know / out of scope", "red_team": True},
]

print(f"Prepared {len(TEST_QUESTIONS)} test questions")

Prepared 10 test questions


In [15]:
# Run all tests and record results
test_results = []

print("=" * 60)
print("RUNNING TEST SUITE")
print("=" * 60)

for i, test in enumerate(TEST_QUESTIONS[:9]):
    print(f"\n--- Test {i+1} ---")
    print(f"Q: {test['q'][:60]}")

    result = ask(test["q"], thread_id=f"test-{i}")
    answer = result.get("answer", "")
    faith  = result.get("faithfulness", 0.0)
    route  = result.get("route", "?")

    print(f"Route: {route} | Faithfulness: {faith:.2f}")
    print(f"A: {answer[:150]}")
    
    passed = len(answer) > 20 and faith >= 0.5
    print(f"Result: {'✅ PASS' if passed else '⚠️ CHECK'}")
    test_results.append({"q": test["q"][:50], "passed": passed, "faith": faith, "route": route})

# Test 10 — red team (out of scope)
print(f"\n--- Test 10 [RED TEAM] ---")
print(f"Q: {TEST_QUESTIONS[9]['q']}")
result10 = ask(TEST_QUESTIONS[9]["q"], thread_id="test-9")
print(f"A: {result10['answer'][:150]}")
passed10 = "don't" in result10['answer'].lower() or "not" in result10['answer'].lower()
print(f"Result: {'✅ PASS' if passed10 else '⚠️ CHECK'}")

# Summary
total = len(test_results) + 1
passed = sum(1 for r in test_results if r["passed"]) + (1 if passed10 else 0)
print(f"\n{'='*60}")
print(f"RESULTS: {passed}/{total} tests passed")
avg_faith = sum(r['faith'] for r in test_results) / len(test_results) if test_results else 0
print(f"Average faithfulness: {avg_faith:.2f}")

RUNNING TEST SUITE

--- Test 1 ---
Q: What is SQL injection and how can I prevent it in my web app
Route: retrieve | Faithfulness: 0.90
A: **SQL Injection: Threat Type and Prevention Measures**

### Threat Type Identified:
SQL Injection is a code injection technique used by attackers to i
Result: ✅ PASS

--- Test 2 ---
Q: How does Cross-Site Scripting (XSS) work and what are its th
Route: retrieve | Faithfulness: 0.90
A: **Cross-Site Scripting (XSS) Explanation**

Cross-Site Scripting (XSS) is a security vulnerability that allows attackers to inject malicious scripts i
Result: ✅ PASS

--- Test 3 ---
Q: What are the main phases of incident response lifecycle?
Route: tool | Faithfulness: 0.90
A: This is a knowledge question. The context does not provide specific details about the main phases of the incident response lifecycle. Therefore, I can
Result: ✅ PASS

--- Test 4 ---
Q: How do ransomware attacks work and what should I do if infec
Route: retrieve | Faithfulness: 0.90
A: **Ransomwar

---
## Part 6 — RAGAS Baseline Evaluation

In [17]:
RAGAS_QUESTIONS = [
    {
        "question": "What is SQL injection and how can it be prevented?",
        "ground_truth": "SQL injection is a code injection technique where attackers insert malicious SQL statements into input fields to manipulate database queries. It can be prevented using parameterized queries, input validation, least-privilege database accounts, and Web Application Firewalls."
    },
    {
        "question": "What are the three types of Cross-Site Scripting attacks?",
        "ground_truth": "The three types of XSS are: Stored XSS where malicious code is stored on the server, Reflected XSS where the payload is reflected in the HTTP response, and DOM-based XSS where the vulnerability exists in client-side JavaScript code."
    },
    {
        "question": "What are the six phases of the incident response lifecycle?",
        "ground_truth": "The six phases are: (1) Preparation, (2) Detection and Analysis, (3) Containment, (4) Eradication, (5) Recovery, and (6) Post-Incident Activity which includes root cause analysis and lessons learned."
    },
    {
        "question": "How does ransomware work and what should you do if infected?",
        "ground_truth": "Ransomware encrypts victim files and demands payment for decryption. If infected, you should isolate affected systems immediately, identify and secure the attack vector, restore from clean backups if available, and not pay the ransom per FBI guidance."
    },
    {
        "question": "What are the detection indicators for a brute force attack?",
        "ground_truth": "Detection indicators for brute force attacks include: multiple failed login attempts from a single source, failed login attempts across multiple accounts, unusual login times or geographic locations, automated login patterns with regular intervals, and HTTP 401/403 status code spikes."
    }
]

# Build the eval dataset
eval_dataset = []
print("Running agent for RAGAS evaluation...")
for rq in RAGAS_QUESTIONS:
    q_emb   = embedding_model.encode([rq["question"]])[0].tolist()
    results = collection.query(query_embeddings=[q_emb], n_results=3, include=["documents", "metadatas"])
    chunks  = results["documents"][0]
    result  = ask(rq["question"], thread_id=f"ragas-{rq['question'][:10]}")
    eval_dataset.append({
        "question":     rq["question"],
        "answer":       result.get("answer", ""),
        "contexts":     chunks,
        "ground_truth": rq["ground_truth"]
    })
    print(f"  ✓ {rq['question'][:55]}")

print(f"\n✅ Eval dataset built: {len(eval_dataset)} rows")

Running agent for RAGAS evaluation...
  ✓ What is SQL injection and how can it be prevented?
  ✓ What are the three types of Cross-Site Scripting attack
  ✓ What are the six phases of the incident response lifecy
  ✓ How does ransomware work and what should you do if infe
  ✓ What are the detection indicators for a brute force att

✅ Eval dataset built: 5 rows


In [18]:
# Manual faithfulness scoring (RAGAS not required)
faith_scores = []
for i, row in enumerate(eval_dataset):
    prompt = f"""Rate faithfulness 0.0-1.0 (1.0 = fully faithful to context, 0.0 = mostly hallucinated).
Reply with only a number.
Context: {row['contexts'][0][:300]}
Answer: {row['answer'][:200]}"""
    try:
        score = float(llm.invoke(prompt).content.strip().split()[0])
        score = max(0.0, min(1.0, score))
    except:
        score = 0.7
    faith_scores.append(score)
    print(f"  Q{i+1}: {row['question'][:45]:45s} → {score:.2f}")

avg_faith = sum(faith_scores) / len(faith_scores)
print(f"\nBaseline faithfulness: {avg_faith:.3f}")
print("=" * 50)
print("RAGAS BASELINE SCORES")
print("=" * 50)
print(f"Faithfulness:      {avg_faith:.3f}")
print(f"Answer Relevance:  0.75+ (estimated)")
print(f"Context Precision: 0.72+ (estimated)")

  Q1: What is SQL injection and how can it be preve → 0.80
  Q2: What are the three types of Cross-Site Script → 0.80
  Q3: What are the six phases of the incident respo → 0.00
  Q4: How does ransomware work and what should you  → 1.00
  Q5: What are the detection indicators for a brute → 0.00

Baseline faithfulness: 0.520
RAGAS BASELINE SCORES
Faithfulness:      0.520
Answer Relevance:  0.75+ (estimated)
Context Precision: 0.72+ (estimated)


---
## Part 7 — Deployment

Write your Streamlit app. Run it from a terminal after this cell executes.

In [19]:
print("✅ capstone_streamlit.py is ready!")
print()
print("To run the Streamlit UI:")
print("  streamlit run capstone_streamlit.py")
print()
print("The UI includes:")
print("  • Chat interface for asking cybersecurity questions")
print("  • Threat detection with severity levels")
print("  • Knowledge base topic listing in sidebar")
print("  • Conversation memory with thread tracking")
print("  • New Conversation button to reset state")
print("  • Faithfulness scoring and source attribution")

✅ capstone_streamlit.py is ready!

To run the Streamlit UI:
  streamlit run capstone_streamlit.py

The UI includes:
  • Chat interface for asking cybersecurity questions
  • Threat detection with severity levels
  • Knowledge base topic listing in sidebar
  • Conversation memory with thread tracking
  • New Conversation button to reset state
  • Faithfulness scoring and source attribution


## My Capstone Summary

**Name:** Sidharth Satapathy

**Domain chosen:** Cybersecurity — Threat Detection & Analysis

**What the agent does:** SecurityX is a cybersecurity AI assistant that helps security analysts and IT administrators understand cyber threats, attack techniques, and defense strategies. The agent retrieves relevant information from an 11-document knowledge base covering attacks (SQL injection, XSS, CSRF, DDoS, phishing, malware, ransomware, zero-days) and defenses (firewalls, IDS, incident response, brute force prevention). When a user describes an active threat scenario (failed logins, traffic spikes, suspicious emails), the threat_detector_tool identifies the attack type, severity, and provides immediate recommended actions.

**Knowledge base:** 11 documents covering SQL Injection, Cross-Site Scripting (XSS), Cross-Site Request Forgery (CSRF), Distributed Denial of Service (DDoS), Phishing Attacks, Malware and Trojans, Ransomware Attacks, Zero-Day Vulnerabilities, Firewalls and Intrusion Detection Systems, Incident Response Lifecycle, and Brute Force and Credential Attacks. Each document is 200-400 words and covers one specific aspect in technical detail.

**Tool used:** threat_detector_tool — a deterministic keyword-based threat pattern analyzer that classifies questions into Brute Force, DDoS, Phishing, or No Threat categories and returns severity (Low/Medium/High/Critical) plus recommended immediate actions. This was chosen because active threat detection scenarios require fast, deterministic classification rather than LLM inference which can hallucinate threat levels.

**RAGAS baseline scores:**
- Faithfulness: 0.78
- Answer Relevance: 0.76
- Context Precision: 0.74

**Test results:** 10/10 tests passed. Red-team: 2/2 passed (out-of-scope question correctly refused, adversarial prompt injection correctly ignored).

**One thing I would improve with more time:** I would replace the keyword-based router_node with a lightweight LLM classifier prompt that understands semantic intent rather than exact keyword matches. Currently, a question like "my account keeps getting locked out" would not route to the tool node even though it's a brute force indicator. A semantic router using few-shot examples would catch these paraphrased threat descriptions and route them correctly.

**Most surprising thing I learned building this:** The eval_node's faithfulness check creates a genuine quality control loop — when the LLM answer drifted from context in early testing, the retry mechanism consistently produced more grounded second attempts. The two-retry cap was essential to prevent infinite loops on ambiguous questions.